In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS rapidapi_cricket_conn 
TYPE HTTP 
OPTIONS (
  host = 'https://ipl-api1.p.rapidapi.com',
  port = 443,
  base_path = '/', 
  bearer_token = 'na'
);

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("rapidapi_cricket_conn")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
dbutils.widgets.text('catalog_name','cricket_dev')
catalog_name=dbutils.widgets.get('catalog_name')
print(catalog_name)

In [0]:
spark.sql(
    f"use catalog {catalog_name}"
)
spark.sql(
    "use schema bronze"
)
spark.sql(
    "create volume if not exists cricket_data"
)
  

In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json
import datetime

# Initialize Workspace Client
w = WorkspaceClient()

# Get Connection Details
conn = w.connections.get("rapidapi_cricket_conn")
base_url = conn.options['host']   

print("Base URL:", base_url)

# Widget for Catalog Name

dbutils.widgets.text('catalog_name','cricket_dev')
catalog_name=dbutils.widgets.get('catalog_name')
print(catalog_name)
     


In [0]:
# Use Catalog and Schema

spark.sql(
    f"use catalog {catalog_name}"
)
spark.sql(
    "use schema bronze"
)
spark.sql(
    "create volume if not exists cricket_data"
)

In [0]:
# API Headers

headers = {
    "X-RapidAPI-Key": "api_key",
    "X-RapidAPI-Host": "api_host"
}


# Fetch Data from RapidAPI
# Checking Endpoint

endpoint = "/match"

url = f"{base_url}{endpoint}"

response = requests.get(url, headers=headers)

if response.status_code != 200:
    raise Exception(f"API Error {response.status_code}: {response.text}")

data = response.json()

print(data)

In [0]:
# Save Data to Volume with Current Date
current_date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

file_path = f"/Volumes/{catalog_name}/bronze/cricket_data/cricket_live_data_{current_date}.json"

dbutils.fs.put(
    file_path, 
    json.dumps(data, indent=2), 
    overwrite=True
)

print(f"Data successfully saved to: {file_path}")